In [1]:
# 질문에 대한 답변
!pip install streamlit sentence-transformers chromadb google-generativeai python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.4/132.4 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 18.5 MB/s eta 0:

In [1]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import google.generativeai as genai


In [3]:
# 지식 자료
knowledge = [
    "사자는 갈귀털이 매우 길다",
    "기린은 목이 길다",
    "치타는 지구상에서 가장 빠르다",
    "하마는 물속에서 생활하는 포유류다",
    "펭귄은 날지 못하나 수영은 잘한다"
]

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(knowledge)
print(embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[[-0.04709733  0.08734604  0.08098347 ...  0.06047381 -0.08786901
  -0.06718885]
 [-0.04074075  0.11778802  0.07432657 ...  0.06017953 -0.05443297
  -0.03423587]
 [-0.01950891  0.07702844 -0.02180516 ... -0.04212037 -0.05715674
   0.01606691]
 [-0.00324782  0.06310052  0.04464058 ...  0.07609498 -0.05634928
   0.00209231]
 [-0.04110395  0.15062107  0.03524347 ...  0.10173291 -0.03646741
   0.00260438]]


In [11]:
# VectorDB ...
chroma_client = chromadb.Client(Settings(persist_directory="./rag_demo", anonymized_telemetry=False))
collection = chroma_client.get_or_create_collection("animals")

for i, (text, emb) in enumerate(zip(knowledge, embeddings)):
    collection.add(
        documents=[text],
        embeddings=[emb.tolist()],
        ids=[f"doc_{i}"]
    )

all_data = collection.get()
print(all_data)

print(collection.count())
doc = collection.get(ids=["doc_0"])
print(doc)

# 질문 처리
query = "목이 긴 동물은?"
query_vec = embedder.encode([query])[0]
# print(query_vec)

results = collection.query(
    query_embeddings=[query_vec.tolist()],
    n_results=3,
    include=["documents"]
)
# print(results)

context = "\n".join(results["documents"][0])
print(f"context :\n{context}")

# LLM에게 프롬프트(검색 + 증강)에 대한 답변을 요구(생성)
import os
from dotenv import load_dotenv
load_dotenv()

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-2.5-flash")

prompt = f"""
  아래 정보를 참고해서 친절한 답을 해줘.
  가능하면 예시나 관련 배경지식도 함께 알려줘.
  정보 :
  {context}
  질문 :
  {query}
  추가사항 : 마크업은 반드시 빼줘.
"""

print(f"prompt:{prompt}")

response = model.generate_content(prompt)
print(f"답변 : {response.text}")

{'ids': ['doc_0', 'doc_1', 'doc_2', 'doc_3', 'doc_4'], 'embeddings': None, 'documents': ['사자는 갈귀털이 매우 길다', '기린은 목이 길다', '치타는 지구상에서 가장 빠르다', '하마는 물속에서 생활하는 포유류다', '펭귄은 날지 못하나 수영은 잘한다'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [None, None, None, None, None]}
5
{'ids': ['doc_0'], 'embeddings': None, 'documents': ['사자는 갈귀털이 매우 길다'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [None]}
context :
기린은 목이 길다
치타는 지구상에서 가장 빠르다
사자는 갈귀털이 매우 길다
prompt:
  아래 정보를 참고해서 친절한 답을 해줘.
  가능하면 예시나 관련 배경지식도 함께 알려줘.
  정보 : 
  기린은 목이 길다
치타는 지구상에서 가장 빠르다
사자는 갈귀털이 매우 길다
  질문 : 
  목이 긴 동물은?
  추가사항 : 마크업은 반드시 빼줘.

답변 : 안녕하세요! 주신 정보를 참고하여 질문에 친절하게 답해드릴게요.

목이 긴 동물은 바로 기린입니다. 주신 정보에도 '기린은 목이 길다'고 정확히 언급되어 있었어요.

기린은 아프리카 초원에서 살며, 길고 우아한 목 덕분에 다른 동물들이 닿기 어려운 높은 나뭇가지의 신선한 잎사귀를 마음껏 먹을 수 있답니다. 또한, 이 긴 목은 주변 환경을 넓게 살피며 다가오는 포식자를 일찍 발견하는 데도 큰 도움을 줍니다. 흥미로운 사실은 기린의 목뼈도 사람과 마찬가지로 7개인데, 각 뼈의 길이가 사람보다 훨씬 길어서 전체 목이 길어지는 것이랍니다. 각 기린마다 독특한 무늬를 가지고 있어 마치 